In [1]:
import tensorflow as tf
import numpy as np
import data_processing as dp
import feature_extraction as fe
import matplotlib.pyplot as plt

In [ ]:
(x_train, y_train), (_, _) = dp.load_data_keras("../Data")
(x_train, y_train), (x_val, y_val) = dp.split_data(x_train, y_train, 1)
(x_train, y_train), (x_val, y_val) = fe.ResnetPreprocess(x_train, y_train, x_val, y_val, sampling=3)

In [2]:
(x_train, y_train), (_, _) = dp.load_data_keras("../Data")
for i in range(3):
    dp.split_data(x_train, y_train, i)

Client 0 - x_train shape: (8250, 32, 32, 3), y_train shape: (8250, 1)
Client 0 - x_val shape: (5000, 32, 32, 3), y_val shape: (5000, 1)
Client 1 - x_train shape: (17250, 32, 32, 3), y_train shape: (17250, 1)
Client 1 - x_val shape: (5000, 32, 32, 3), y_val shape: (5000, 1)
Client 2 - x_train shape: (19500, 32, 32, 3), y_train shape: (19500, 1)
Client 2 - x_val shape: (5000, 32, 32, 3), y_val shape: (5000, 1)


In [ ]:
def poison_dataset(x_train, y_train, target_label, desired_label, poison_ratio=0.1):
    """
    Poison the training dataset by changing a fraction of target_label to the desired label.
    
    Args:
        x_train (numpy.ndarray): Training images.
        y_train (numpy.ndarray): Training labels.
        target_label (int): The label to poison.
        desired_label (int): The label to assign to poisoned samples.
        poison_ratio (float): Fraction of the dataset to poison.
        preprocess_func (callable, optional): Preprocessing function from fe.ResnetPreprocess.
        
    Returns:
        Tuple[numpy.ndarray, numpy.ndarray]: Poisoned training images and labels.
    """
    target_indices = np.where(y_train == target_label)[0]
    num_available = len(target_indices)
    num_poison = int(len(y_train) * poison_ratio)
    num_poison = min(num_poison, num_available)
    
    if num_poison == 0:
        return x_train, y_train
    
    poison_indices = np.random.choice(target_indices, num_poison, replace=False)
    y_train_poisoned = np.copy(y_train)
    y_train_poisoned[poison_indices] = desired_label
    x_train_poisoned = np.copy(x_train)
    return x_train_poisoned, y_train_poisoned

In [ ]:
poison_dataset(x_train, y_train, 1, 0, poison_ratio=0.1)

In [ ]:
accuracy_history = [0.4458000063896179, 0.5960000157356262, 0.5893999934196472, 0.6269999742507935, 0.6790000200271606, 0.5388000011444092, 0.6132000088691711, 0.6549999713897705, 0.6516000032424927, 0.6425999999046326, 0.5838000178337097, 0.6796000003814697, 0.704200029373169, 0.6624000072479248, 0.6638000011444092, 0.7020000219345093, 0.6711999773979187, 0.6693999767303467, 0.6741999983787537, 0.7102000117301941, 0.7149999737739563, 0.7268000245094299, 0.770799994468689, 0.7379999756813049, 0.7666000127792358, 0.7311999797821045, 0.7814000248908997, 0.7210000157356262, 0.725600004196167, 0.7657999992370605, 0.7364000082015991, 0.7684000134468079, 0.7282000184059143, 0.7598000168800354, 0.7310000061988831, 0.7674000263214111, 0.7458000183105469, 0.7501999735832214, 0.7620000243186951, 0.7035999894142151, 0.7657999992370605, 0.7283999919891357, 0.7710000276565552, 0.7468000054359436, 0.7793999910354614, 0.7483999729156494, 0.7853999733924866, 0.7558000087738037, 0.7483999729156494, 0.743399977684021, 0.7752000093460083, 0.7501999735832214, 0.7760000228881836, 0.7613999843597412, 0.7728000283241272, 0.7742000222206116, 0.7554000020027161, 0.7541999816894531, 0.7688000202178955, 0.7766000032424927, np.array([-0.15755312, -0.15183131, -0.08264791, -0.04522571, -0.10615061,-0.19820455, -0.12422583, -0.23476243, -0.12649502, -0.25884172], dtype=np.float32)]

In [ ]:
accuracy_history[:-1]

In [ ]:
plt.figure()
plt.plot(range(1, len(accuracy_history[:-1]) + 1), accuracy_history[:-1], label='Validation Accuracy')
plt.title('Server Validation Accuracy Over Rounds')
plt.xlabel('Round')
plt.ylabel('Accuracy')
plt.legend()
plt.savefig('server_validation_accuracy.png')
plt.close()